<a href="https://colab.research.google.com/github/emiliaarias-png/hackaton-iia-uade-2026/blob/main/sistema_M04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema M04 · ¿Este título va a la portada del usuario?
Pista: **Neurona** (perceptrón con pesos y sesgo). Integrantes: Pilar Medina Livigni · María Emilia Arias.

In [1]:
# M04 · ¿Este título va a la portada del usuario?
# Datos: los 20 casos de la ficha M04 (también en datos/M04.csv). Casos 1 a 14 = entrenamiento, 15 a 20 = prueba.
import numpy as np

# columnas: afinidad (0 a 1), duración (min), calificación (0 a 10), clics esperados (%), resultado real (1 = SÍ)
datos = np.array([
 [0.33,115,4.2,22,0],[0.75,172,9.1, 3,1],[0.57,169,3.9, 9,0],[0.26, 55,7.3, 4,1],[0.14,164,4.0,23,0],
 [0.64, 69,5.0,17,1],[0.71, 34,8.4, 2,1],[0.66,140,7.7, 3,1],[0.96, 27,9.4,20,1],[0.59, 50,7.6,21,1],
 [0.03,176,5.3,22,0],[0.19, 61,6.2,17,0],[0.11,106,6.4,24,0],[0.71,158,5.5,24,0],
 [0.83, 38,9.3, 4,1],[0.48,135,6.3,19,0],[0.35,103,5.4,18,0],[0.34,116,9.3, 8,0],[0.98, 92,3.3,20,1],[0.71, 55,3.9,17,0],
])
ent, pru = datos[:14], datos[14:]
y_ent, y_pru = ent[:, 4].astype(int), pru[:, 4].astype(int)
print("Entrenamiento:", len(ent), "casos | Prueba:", len(pru), "casos")


Entrenamiento: 14 casos | Prueba: 6 casos


In [2]:
# LA NEURONA (perceptrón): n = w1·afinidad + w2·duración(h) + w3·calificación + b ; SÍ si n >= u
# Entradas que usamos: afinidad, duración (en horas) y calificación. NO usamos "clics esperados" (ver README, sección 1).
w = np.array([5.0, -1.0, 1.0])   # pesos
b = -6.7                          # sesgo
u = 0.0                           # umbral

def entradas(X):
    # X tiene columnas: afinidad, duración (min), calificación, ...
    return np.column_stack([X[:, 0], X[:, 1] / 60.0, X[:, 2]])

def n_neta(X):
    return entradas(X) @ w + b

def predecir(X):
    return (n_neta(X) >= u).astype(int)   # 1 = SÍ (va a portada), 0 = NO

print("n del entrenamiento:", np.round(n_neta(ent), 2))
print("n de la prueba     :", np.round(n_neta(pru), 2))


n del entrenamiento: [-2.77  3.28 -2.77  0.98 -4.73  0.35  4.68  1.97  7.05  3.02 -4.18 -0.57
 -1.52 -0.28]
n de la prueba     : [ 6.12 -0.25 -1.27  2.37 -0.03 -0.17]


In [3]:
# MATRIZ DE CONFUSIÓN Y MÉTRICAS
def matriz(pred, real):
    VP = int(((pred == 1) & (real == 1)).sum()); FN = int(((pred == 0) & (real == 1)).sum())
    FP = int(((pred == 1) & (real == 0)).sum()); VN = int(((pred == 0) & (real == 0)).sum())
    return VP, FN, FP, VN

def metricas(VP, FN, FP, VN):
    total = VP + FN + FP + VN
    div = lambda a, c: a / c if c else float("nan")      # nan = indefinida (denominador 0)
    return {"exactitud": (VP + VN) / total, "precision": div(VP, VP + FP),
            "sensibilidad": div(VP, VP + FN), "especificidad": div(VN, VN + FP)}

for nombre, X, y in [("ENTRENAMIENTO", ent, y_ent), ("PRUEBA", pru, y_pru)]:
    VP, FN, FP, VN = matriz(predecir(X), y)
    print(nombre, "| VP =", VP, "FN =", FN, "FP =", FP, "VN =", VN, "| suma =", VP + FN + FP + VN, "| SÍ reales =", VP + FN)
    for k, v in metricas(VP, FN, FP, VN).items():
        print("   ", k, "=", round(100 * v, 1), "%")


ENTRENAMIENTO | VP = 7 FN = 0 FP = 0 VN = 7 | suma = 14 | SÍ reales = 7
    exactitud = 100.0 %
    precision = 100.0 %
    sensibilidad = 100.0 %
    especificidad = 100.0 %
PRUEBA | VP = 1 FN = 1 FP = 1 VN = 3 | suma = 6 | SÍ reales = 2
    exactitud = 66.7 %
    precision = 50.0 %
    sensibilidad = 50.0 %
    especificidad = 75.0 %


In [4]:
# LÍNEA BASE TRIVIAL: decir siempre "NO" (la clase más frecuente en entrenamiento)
for nombre, y in [("entrenamiento", y_ent), ("prueba", y_pru)]:
    VP, FN, FP, VN = matriz(np.zeros(len(y), dtype=int), y)
    print("Línea base en", nombre, ": exactitud =", round(100 * (VP + VN) / len(y), 1), "% | sensibilidad =", round(100 * VP / (VP + FN), 1), "%")

# Comparación: la regla que premia el equipo de producto ("SÍ si clics esperados >= 15 %")
for nombre, X, y in [("entrenamiento", ent, y_ent), ("prueba", pru, y_pru)]:
    p = (X[:, 3] >= 15).astype(int)
    print("Regla por clics en", nombre, ":", matriz(p, y), "exactitud =", round(100 * (p == y).mean(), 1), "%")


Línea base en entrenamiento : exactitud = 50.0 % | sensibilidad = 0.0 %
Línea base en prueba : exactitud = 66.7 % | sensibilidad = 0.0 %
Regla por clics en entrenamiento : (3, 4, 6, 1) exactitud = 28.6 %
Regla por clics en prueba : (1, 1, 3, 1) exactitud = 33.3 %


In [5]:
# CASO EN VIVO: cargá un caso nuevo y el sistema responde SÍ o NO
def decidir(afinidad, duracion_min, calificacion):
    n = w[0] * afinidad + w[1] * (duracion_min / 60) + w[2] * calificacion + b
    return ("SÍ" if n >= u else "NO"), round(float(n), 2)

print(decidir(afinidad=0.80, duracion_min=90, calificacion=8.0))   # ejemplo: cambiá los valores


('SÍ', 3.8)
